In [11]:
!pip install -q dagshub mlflow xgboost

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
import xgboost as xgb
import mlflow
import mlflow.sklearn
import dagshub
from kaggle_secrets import UserSecretsClient
import os
import warnings
warnings.filterwarnings('ignore')

# MLflow setup
user_secrets = UserSecretsClient()
MLFLOW_TRACKING_PASSWORD = user_secrets.get_secret("ML_token1")
MLFLOW_TRACKING_USERNAME = user_secrets.get_secret("username")

repo_name = "IEEE-Fraud-Detection"
mlflow.set_tracking_uri(f"https://dagshub.com/{MLFLOW_TRACKING_USERNAME}/{repo_name}.mlflow")
os.environ["MLFLOW_TRACKING_USERNAME"] = MLFLOW_TRACKING_USERNAME
os.environ["MLFLOW_TRACKING_PASSWORD"] = MLFLOW_TRACKING_PASSWORD

dagshub.init(repo_name=repo_name, repo_owner=MLFLOW_TRACKING_USERNAME, mlflow=True)
mlflow.set_experiment("XGBoost_Training")
print("✅ MLflow connected - Experiment: XGBoost_Training")

Initialized MLflow to track repo "mgior23/IEEE-Fraud-Detection"

Repository mgior23/IEEE-Fraud-Detection initialized!

✅ MLflow connected - Experiment: XGBoost_Training


# Cleaning

In [12]:
# Load
train_trans = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv')
train_id = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv')
test_trans = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv')
test_id = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv')

# Merge
train = train_trans.merge(train_id, on='TransactionID', how='left')
test = test_trans.merge(test_id, on='TransactionID', how='left')

# Separate target
y = train['isFraud']
train_ids = train['TransactionID']
test_ids = test['TransactionID']

# Drop ID and target from features
X_train = train.drop(['TransactionID', 'isFraud'], axis=1)
X_test = test.drop(['TransactionID'], axis=1)

print(f"Train shape: {X_train.shape}")
print(f"Test shape: {X_test.shape}")
print(f"Fraud rate: {y.mean():.2%}")

# Log cleaning info
with mlflow.start_run(run_name="XGBoost_Cleaning"):
    mlflow.log_params({
        "train_samples": len(X_train),
        "test_samples": len(X_test),
        "fraud_rate": round(y.mean(), 4),
        "n_features": X_train.shape[1]
    })

Train shape: (590540, 432)
Test shape: (506691, 432)
Fraud rate: 3.50%
🏃 View run XGBoost_Cleaning at: https://dagshub.com/mgior23/IEEE-Fraud-Detection.mlflow/#/experiments/0/runs/1a7ace50a2794309941e9afc3e330ce8
🧪 View experiment at: https://dagshub.com/mgior23/IEEE-Fraud-Detection.mlflow/#/experiments/0


In [13]:
import os
for root, dirs, files in os.walk('/kaggle/input/competitions'):
    for file in files:
        print(os.path.join(root, file))

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [14]:
# Drop columns with >90% missing
missing_pct = X_train.isnull().sum() / len(X_train)
drop_cols = missing_pct[missing_pct > 0.9].index.tolist()
print(f"Dropping {len(drop_cols)} columns with >90% missing")
X_train = X_train.drop(drop_cols, axis=1, errors='ignore')
X_test = X_test.drop(drop_cols, axis=1, errors='ignore')

# Drop high cardinality categorical columns (>100 unique values)
cat_cols = X_train.select_dtypes(include=['object']).columns
high_card = [col for col in cat_cols if X_train[col].nunique() > 100]
print(f"Dropping {len(high_card)} high-cardinality columns")
X_train = X_train.drop(high_card, axis=1, errors='ignore')
X_test = X_test.drop(high_card, axis=1, errors='ignore')

# Align columns — ensure train and test have the same columns after dropping
X_test = X_test.reindex(columns=X_train.columns)

print(f"Final train shape: {X_train.shape}")
print(f"Final test shape: {X_test.shape}")

Dropping 12 columns with >90% missing
Dropping 3 high-cardinality columns
Final train shape: (590540, 417)
Final test shape: (506691, 417)


# Feature Engineering

In [15]:
# V-columns (anonymized features) - keep as is for now
# C-columns (counting features) - keep as is
# D-columns (timedelta features) - keep as is
# M-columns (match features) - binary, keep as is

# TransactionAmt - log transform
X_train['TransactionAmt_log'] = np.log1p(X_train['TransactionAmt'])
X_test['TransactionAmt_log'] = np.log1p(X_test['TransactionAmt'])

# Time features from TransactionDT (seconds from reference point)
X_train['TransactionDT_day'] = X_train['TransactionDT'] // (24*3600)
X_train['TransactionDT_hour'] = (X_train['TransactionDT'] % (24*3600)) // 3600

X_test['TransactionDT_day'] = X_test['TransactionDT'] // (24*3600)
X_test['TransactionDT_hour'] = (X_test['TransactionDT'] % (24*3600)) // 3600

# Aggregate features
agg_features = []
for c in ['C1','C2','C3','C4']:
    if c in X_train.columns:
        X_train[f'{c}_to_amt'] = X_train[c] / (X_train['TransactionAmt'] + 1)
        X_test[f'{c}_to_amt'] = X_test[c] / (X_test['TransactionAmt'] + 1)
        agg_features.append(f'{c}_to_amt')

print(f"Created {len(agg_features)} ratio features")
print(f"New shape: {X_train.shape}")

with mlflow.start_run(run_name="XGBoost_Feature_Engineering"):
    mlflow.log_params({
        "n_features_after_eng": X_train.shape[1],
        "new_features": len(agg_features) + 2  # log + time features
    })

Created 4 ratio features
New shape: (590540, 424)
🏃 View run XGBoost_Feature_Engineering at: https://dagshub.com/mgior23/IEEE-Fraud-Detection.mlflow/#/experiments/0/runs/ddadb0d564844fb1b33bf6eb6c862f22
🧪 View experiment at: https://dagshub.com/mgior23/IEEE-Fraud-Detection.mlflow/#/experiments/0


# Feature Selection

In [16]:
# Separate categorical and numerical
cat_cols = X_train.select_dtypes(include=['object']).columns.tolist()
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()

print(f"Categorical: {len(cat_cols)}")
print(f"Numerical: {len(num_cols)}")

# Label encode categoricals
le_dict = {}
for col in cat_cols:
    le = LabelEncoder()
    X_train[col] = X_train[col].astype(str)
    X_test[col] = X_test[col].astype(str)
    
    # Fit on train, handle unseen in test
    X_train[col] = le.fit_transform(X_train[col])
    X_test[col] = X_test[col].map(lambda x: le.transform([x])[0] if x in le.classes_ else -1)
    le_dict[col] = le

print("✅ Categorical encoding done")

Categorical: 26
Numerical: 398
✅ Categorical encoding done


# Training

In [17]:
# Baseline XGBoost
params_v1 = {
    'n_estimators': 300,
    'max_depth': 6,
    'learning_rate': 0.05,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'random_state': 42,
    'tree_method': 'hist',
    'eval_metric': 'auc'
}

model_v1 = xgb.XGBClassifier(**params_v1)

# 5-Fold CV
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y)):
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    model_v1.fit(X_tr, y_tr)
    preds = model_v1.predict_proba(X_val)[:, 1]
    auc = roc_auc_score(y_val, preds)
    cv_scores.append(auc)
    print(f"Fold {fold+1} AUC: {auc:.5f}")

mean_auc = np.mean(cv_scores)
std_auc = np.std(cv_scores)
print(f"\n✅ Mean CV AUC: {mean_auc:.5f} (+/- {std_auc:.5f})")

# Log to MLflow
with mlflow.start_run(run_name="XGBoost_v1_baseline"):
    mlflow.log_params(params_v1)
    mlflow.log_metrics({
        "cv_auc_mean": mean_auc,
        "cv_auc_std": std_auc
    })
    mlflow.sklearn.log_model(model_v1, "model")

Fold 1 AUC: 0.93342
Fold 2 AUC: 0.93506
Fold 3 AUC: 0.92932
Fold 4 AUC: 0.93077
Fold 5 AUC: 0.93274

✅ Mean CV AUC: 0.93226 (+/- 0.00202)


2026/05/04 16:19:29 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/04 16:19:30 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run XGBoost_v1_baseline at: https://dagshub.com/mgior23/IEEE-Fraud-Detection.mlflow/#/experiments/0/runs/021643d6c4bc4eb9a712719451b6d4f8
🧪 View experiment at: https://dagshub.com/mgior23/IEEE-Fraud-Detection.mlflow/#/experiments/0


In [18]:
params_v2 = {
    'n_estimators': 500,
    'max_depth': 4,
    'learning_rate': 0.03,
    'subsample': 0.7,
    'colsample_bytree': 0.7,
    'min_child_weight': 3,
    'reg_alpha': 0.1,
    'reg_lambda': 1.5,
    'random_state': 42,
    'tree_method': 'hist',
    'eval_metric': 'auc'
}

model_v2 = xgb.XGBClassifier(**params_v2)

cv_scores_v2 = []
for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y)):
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    model_v2.fit(X_tr, y_tr)
    preds = model_v2.predict_proba(X_val)[:, 1]
    auc = roc_auc_score(y_val, preds)
    cv_scores_v2.append(auc)
    print(f"Fold {fold+1} AUC: {auc:.5f}")

mean_auc_v2 = np.mean(cv_scores_v2)
std_auc_v2 = np.std(cv_scores_v2)
print(f"\n✅ Mean CV AUC: {mean_auc_v2:.5f} (+/- {std_auc_v2:.5f})")

with mlflow.start_run(run_name="XGBoost_v2_tuned"):
    mlflow.log_params(params_v2)
    mlflow.log_metrics({
        "cv_auc_mean": mean_auc_v2,
        "cv_auc_std": std_auc_v2
    })
    mlflow.sklearn.log_model(model_v2, "model")

Fold 1 AUC: 0.91110
Fold 2 AUC: 0.91147
Fold 3 AUC: 0.90535
Fold 4 AUC: 0.90807
Fold 5 AUC: 0.91022

✅ Mean CV AUC: 0.90924 (+/- 0.00228)


2026/05/04 16:26:03 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/04 16:26:04 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run XGBoost_v2_tuned at: https://dagshub.com/mgior23/IEEE-Fraud-Detection.mlflow/#/experiments/0/runs/034f7ab291ee42489d8254e28fe81243
🧪 View experiment at: https://dagshub.com/mgior23/IEEE-Fraud-Detection.mlflow/#/experiments/0


In [19]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
import pickle

# Train final model on full data
best_params = params_v2 if mean_auc_v2 > mean_auc else params_v1
best_run_name = "XGBoost_v2_tuned" if mean_auc_v2 > mean_auc else "XGBoost_v1_baseline"

final_model = xgb.XGBClassifier(**best_params)
final_model.fit(X_train, y)

# Save preprocessing info
preprocessing_info = {
    'drop_cols': drop_cols,
    'high_card': high_card,
    'cat_cols': cat_cols,
    'le_dict': le_dict,
    'feature_cols': X_train.columns.tolist()
}
with open('/kaggle/working/xgboost_preprocessing.pkl', 'wb') as f:
    pickle.dump(preprocessing_info, f)

# Log final model to MLflow and register it
with mlflow.start_run(run_name="XGBoost_Final_Model") as run:
    mlflow.log_params(best_params)
    mlflow.log_metrics({
        "final_cv_auc_mean": max(mean_auc, mean_auc_v2),
        "best_version": 2 if mean_auc_v2 > mean_auc else 1
    })
    mlflow.log_param("best_run", best_run_name)
    
    # Log preprocessing info as artifact
    mlflow.log_artifact('/kaggle/working/xgboost_preprocessing.pkl')
    
    # Log and register the model in Model Registry
    mlflow.sklearn.log_model(
        final_model,
        artifact_path="xgboost_final_pipeline",
        registered_model_name="XGBoost_FraudDetection"
    )
    
    run_id = run.info.run_id
    print(f"✅ Run ID: {run_id}")

print(f"✅ Best run: {best_run_name}")
print(f"✅ Best CV AUC: {max(mean_auc, mean_auc_v2):.5f}")
print("✅ Model registered in MLflow Model Registry as 'XGBoost_FraudDetection'")

✅ Model and preprocessing saved
Best CV AUC: 0.93226


In [27]:
import xgboost as xgb
import mlflow
import mlflow.xgboost

mlflow.set_tracking_uri("https://dagshub.com/mgior23/IEEE-Fraud-Detection.mlflow")

# Train final model on full training data
final_model = xgb.XGBClassifier(**params_v1)  # v1 had best AUC
final_model.fit(X_train, y)

# Register it properly
with mlflow.start_run(run_name="XGBoost_Final_Model") as run:
    mlflow.log_params(params_v1)
    mlflow.log_metric("cv_auc_mean", mean_auc)
    
    # Save preprocessing as artifact
    import pickle
    with open('/kaggle/working/xgboost_preprocessing.pkl', 'wb') as f:
        pickle.dump(preprocessing_info, f)
    mlflow.log_artifact('/kaggle/working/xgboost_preprocessing.pkl')
    
    # Log model with explicit artifact path
    mlflow.xgboost.log_model(
        final_model,
        artifact_path="xgboost_final_pipeline",
        registered_model_name="XGBoost_FraudDetection"
    )
    
    final_run_id = run.info.run_id
    print(f"✅ Run ID: {final_run_id}")
    print("✅ Model saved and registered!")

2026/05/04 16:35:45 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
Registered model 'XGBoost_FraudDetection' already exists. Creating a new version of this model...
2026/05/04 16:35:54 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: XGBoost_FraudDetection, version 1
Created version '1' of model 'XGBoost_FraudDetection'.


✅ Run ID: 86cc597066614a0c8bb1f82985918238
✅ Model saved and registered!
🏃 View run XGBoost_Final_Model at: https://dagshub.com/mgior23/IEEE-Fraud-Detection.mlflow/#/experiments/0/runs/86cc597066614a0c8bb1f82985918238
🧪 View experiment at: https://dagshub.com/mgior23/IEEE-Fraud-Detection.mlflow/#/experiments/0
